# 01 -- Data Exploration

**Contact Luck Prototype v0.1**

Scope: explore the cleaned fair-batted-ball event table produced by `mlb_luck_score.data.clean_batted_balls` / `mlb_luck_score.data.clean_development_data`. This notebook is descriptive only -- it does not fit any model.

Prefers the full 2021-2024 development dataset (`cleaned_development_data.parquet`) when available, and falls back to the smaller one-week bootstrap sample (`cleaned_batted_balls.parquet`) otherwise.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)

## Load data

If this cell reports missing data, run one of the following from the repository root first:

```bash
make download-sample && make clean-data                       # small one-week 2024 sample
make download-development-data && make clean-development-data # full 2021-2024 dataset (slower, needs approval -- see README)
```

In [ ]:
from mlb_luck_score.config import DEVELOPMENT_SEASONS, PROCESSED_DATA_DIR

DEVELOPMENT_PATH = PROCESSED_DATA_DIR / "cleaned_development_data.parquet"
SAMPLE_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"

if DEVELOPMENT_PATH.exists():
    df = pd.read_parquet(DEVELOPMENT_PATH)
    print(f"Loaded the full development dataset: {len(df)} rows from {DEVELOPMENT_PATH}")
elif SAMPLE_PATH.exists():
    df = pd.read_parquet(SAMPLE_PATH)
    print(
        f"Full development dataset not found at {DEVELOPMENT_PATH}.\n"
        f"Falling back to the one-week bootstrap sample: {len(df)} rows from {SAMPLE_PATH}.\n"
        "Run `make download-development-data` then `make clean-development-data` for the "
        "full 2021-2024 dataset."
    )
else:
    df = None
    print(
        "No cleaned data found. Run one of:\n"
        "  make download-sample && make clean-data                     # small one-week sample\n"
        "  make download-development-data && make clean-development-data # full 2021-2024 dataset\n"
        "then re-run this notebook."
    )

## Outcome counts

In [ ]:
if df is not None:
    display(df["outcome_class"].value_counts(dropna=False))
else:
    print("Skipped -- no data loaded.")

## Missingness by column

In [ ]:
if df is not None:
    display(df.isna().mean().sort_values(ascending=False).to_frame("missing_fraction"))
else:
    print("Skipped -- no data loaded.")

## Eligible vs. training-eligible counts

In [ ]:
if df is not None:
    print("is_eligible (fair batted ball, Version 0.1 event list):")
    display(df["is_eligible"].value_counts())
    print("\neligible_for_training:")
    display(df["eligible_for_training"].value_counts())
    print("\ntraining_exclusion_reason (for rows excluded from training):")
    display(df.loc[~df["eligible_for_training"], "training_exclusion_reason"].value_counts())
else:
    print("Skipped -- no data loaded.")

## Summary by season

Only meaningful when multiple seasons are loaded (i.e. the full development dataset). With just the one-week sample, this will show a single season.

In [ ]:
if df is not None and "season" in df.columns:
    season_summary = df.groupby("season", dropna=True).agg(
        total_rows=("event_id", "count"),
        eligible_for_training=("eligible_for_training", "sum"),
    )
    season_summary["excluded_from_training"] = (
        season_summary["total_rows"] - season_summary["eligible_for_training"]
    )
    display(season_summary)

    print("\nOutcome class counts by season:")
    display(pd.crosstab(df["season"], df["outcome_class"], dropna=True))
else:
    print("Skipped -- no data loaded.")

## Contact-variable distributions

In [ ]:
if df is not None:
    display(df[["launch_speed", "launch_angle", "hit_distance_sc"]].describe())
else:
    print("Skipped -- no data loaded.")

## Outcome balance (bar chart)

In [ ]:
if df is not None:
    import matplotlib.pyplot as plt

    df["outcome_class"].value_counts().plot(kind="bar", title="Outcome class frequency")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped -- no data loaded.")